# Surf Equipment Database

## Overview

This notebook performs the following tasks:

1. Load the surf equipment workbook and identify its master and transactional tables.
2. Create a SQLite database with primary keys and foreign-key constraints.
3. Validate table structure, row counts, keys, and referential integrity.
4. Generate an entity relationship diagram in DOT, SVG, and PNG formats.
5. Build fact sheets for every board and every rig.
6. Export all fact sheets as PNG images for easy sharing.

### Task 1: Load the workbook

This cell imports the required libraries, reads the Excel workbook, identifies the master and transactional tables, and records each table's primary key.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

WORKBOOK = Path('surf_equipment.xlsx')

if not WORKBOOK.exists():
    raise FileNotFoundError(WORKBOOK.resolve())

workbook = pd.ExcelFile(WORKBOOK)
sheet_names = workbook.sheet_names
if len(sheet_names) != 11:
    raise ValueError(f'Expected 11 sheets, found {len(sheet_names)}')
master_names = sheet_names[:9]
transaction_names = sheet_names[9:]
tables = {name: pd.read_excel(WORKBOOK, sheet_name=name) for name in sheet_names}
primary_keys = {name: table.columns[0] for name, table in tables.items()}
primary_key_to_table = {key: name for name, key in primary_keys.items()}

print('Master tables:', master_names)
print('Transactional tables:', transaction_names)
print('Primary keys:', primary_keys)

Master tables: ['sails', 'masts', 'extensions', 'base_pulleys', 'booms', 'rdm_sdm_shims', 'uphauls', 'harness_lines', 'boards']
Transactional tables: ['rigs', 'rig_board']
Primary keys: {'sails': 'sail_id', 'masts': 'mast_id', 'extensions': 'extension_id', 'base_pulleys': 'base_pulley_id', 'booms': 'boom_id', 'rdm_sdm_shims': 'rdm_sdm_shim_id', 'uphauls': 'uphaul_id', 'harness_lines': 'harness_line_id', 'boards': 'board_id', 'rigs': 'rig_id', 'rig_board': 'rig_board_id'}


### Task 2: Create the SQLite database

This cell defines the SQL helpers, creates the tables with primary and foreign keys, loads the workbook data, and commits the database.

In [2]:
DATABASE = Path('surf_equipment.db')


def quote_identifier(identifier):
    quote = chr(34)
    return quote + str(identifier).replace(quote, quote + quote) + quote


def sqlite_type(series):
    if pd.api.types.is_integer_dtype(series):
        return 'INTEGER'
    if pd.api.types.is_numeric_dtype(series):
        return 'REAL'
    return 'TEXT'


def python_value(value):
    if pd.isna(value):
        return None
    return value.item() if hasattr(value, 'item') else value


def create_table_sql(table_name, table, transaction_names, primary_key_to_table, primary_keys):
    primary_key = table.columns[0]
    definitions = []
    for column in table.columns:
        definition = f'{quote_identifier(column)} {sqlite_type(table[column])}'
        if column == primary_key:
            definition += ' PRIMARY KEY'
        definitions.append(definition)
    if table_name in transaction_names:
        for column in table.columns[1:]:
            parent_table = primary_key_to_table.get(column)
            if parent_table and parent_table != table_name:
                parent_key = primary_keys[parent_table]
                definitions.append(f'FOREIGN KEY ({quote_identifier(column)}) REFERENCES {quote_identifier(parent_table)} ({quote_identifier(parent_key)})')
    body = ',\n    '.join(definitions)
    return f'CREATE TABLE {quote_identifier(table_name)} (\n    {body}\n);'


connection = sqlite3.connect(DATABASE)
connection.execute('PRAGMA foreign_keys = ON')
for table_name in reversed(sheet_names):
    connection.execute(f'DROP TABLE IF EXISTS {quote_identifier(table_name)}')
for table_name in sheet_names:
    connection.execute(
        create_table_sql(
            table_name,
            tables[table_name],
            transaction_names,
            primary_key_to_table,
            primary_keys,
        )
    )
for table_name, table in tables.items():
    columns = ', '.join(quote_identifier(column) for column in table.columns)
    placeholders = ', '.join('?' for _ in table.columns)
    rows = [[python_value(value) for value in row] for row in table.itertuples(index=False, name=None)]
    connection.executemany(f'INSERT INTO {quote_identifier(table_name)} ({columns}) VALUES ({placeholders})', rows)
connection.commit()
print(f'Created {DATABASE.resolve()}')

Created /workspaces/surf_equipment/surf_equipment.db


### Task 3: Validate the database

This cell checks primary keys, row counts, foreign-key definitions, and SQLite referential integrity.

In [3]:
for table_name in sheet_names:
    info = connection.execute(f'PRAGMA table_info({quote_identifier(table_name)})').fetchall()
    assert [row[1] for row in info if row[5] == 1] == [primary_keys[table_name]]
    assert connection.execute(f'SELECT COUNT(*) FROM {quote_identifier(table_name)}').fetchone()[0] == len(tables[table_name])
foreign_keys = []
for table_name in transaction_names:
    foreign_keys.extend(connection.execute(f'PRAGMA foreign_key_list({quote_identifier(table_name)})').fetchall())
assert foreign_keys, 'No foreign keys were created'
assert connection.execute('PRAGMA foreign_key_check').fetchall() == []
print(f'Validated {len(sheet_names)} tables and {len(foreign_keys)} foreign-key constraints.')
for table_name in transaction_names:
    print(table_name, connection.execute(f'PRAGMA foreign_key_list({quote_identifier(table_name)})').fetchall())

Validated 11 tables and 10 foreign-key constraints.
rigs [(0, 0, 'harness_lines', 'harness_line_id', 'harness_line_id', 'NO ACTION', 'NO ACTION', 'NONE'), (1, 0, 'uphauls', 'uphaul_id', 'uphaul_id', 'NO ACTION', 'NO ACTION', 'NONE'), (2, 0, 'rdm_sdm_shims', 'rdm_sdm_shim_id', 'rdm_sdm_shim_id', 'NO ACTION', 'NO ACTION', 'NONE'), (3, 0, 'booms', 'boom_id', 'boom_id', 'NO ACTION', 'NO ACTION', 'NONE'), (4, 0, 'extensions', 'extension_id', 'extension_id', 'NO ACTION', 'NO ACTION', 'NONE'), (5, 0, 'base_pulleys', 'base_pulley_id', 'base_pulley_id', 'NO ACTION', 'NO ACTION', 'NONE'), (6, 0, 'masts', 'mast_id', 'mast_id', 'NO ACTION', 'NO ACTION', 'NONE'), (7, 0, 'sails', 'sail_id', 'sail_id', 'NO ACTION', 'NO ACTION', 'NONE')]
rig_board [(0, 0, 'rigs', 'rig_id', 'rig_id', 'NO ACTION', 'NO ACTION', 'NONE'), (1, 0, 'boards', 'board_id', 'board_id', 'NO ACTION', 'NO ACTION', 'NONE')]


### Task 4: Generate the entity relationship diagram

This section builds the DOT representation of the database schema and exports DOT, SVG, and PNG files to `output/ERD` when the Graphviz executable is available.

In [4]:
import shutil

ERD_OUTPUT_DIR = Path('output') / 'ERD'
ERD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ERD_DOT = ERD_OUTPUT_DIR / 'surf_equipment_erd.dot'
ERD_SVG = ERD_OUTPUT_DIR / 'surf_equipment_erd.svg'
ERD_PNG = ERD_OUTPUT_DIR / 'surf_equipment_erd.png'


def dot_id(value):
    return chr(34) + str(value).replace(chr(34), chr(92) + chr(34)) + chr(34)


def is_unique_column(table_name, column_name):
    table_info = connection.execute(f'PRAGMA table_info({quote_identifier(table_name)})').fetchall()
    if any(row[1] == column_name and row[5] == 1 for row in table_info):
        return True
    indexes = connection.execute(f'PRAGMA index_list({quote_identifier(table_name)})').fetchall()
    for index in indexes:
        index_name, is_unique = index[1], index[2]
        if is_unique:
            index_columns = connection.execute(f'PRAGMA index_info({quote_identifier(index_name)})').fetchall()
            if [row[2] for row in index_columns] == [column_name]:
                return True
    return False


dot_lines = ['digraph surf_equipment {', '  graph [rankdir=LR];', '  node [shape=plain];', '  edge [fontname="Helvetica"];']
for table_name in sheet_names:
    fields = connection.execute(f'PRAGMA table_info({quote_identifier(table_name)})').fetchall()
    rows = [f'<TR><TD BGCOLOR="#DDEBF7"><B>{table_name}</B></TD></TR>']
    for field in fields:
        marker = 'PK' if field[5] else ''
        rows.append(f'<TR><TD ALIGN="LEFT">{marker} {field[1]} : {field[2]}</TD></TR>')
    label = '<<TABLE BORDER="1" CELLBORDER="0" CELLSPACING="0" CELLPADDING="5">' + ''.join(rows) + '</TABLE>>'
    dot_lines.append(f'  {dot_id(table_name)} [label={label}];')
for table_name in transaction_names:
    for foreign_key in connection.execute(f'PRAGMA foreign_key_list({quote_identifier(table_name)})'):
        parent_table, parent_column, child_column = foreign_key[2], foreign_key[4], foreign_key[3]
        child_cardinality = '1' if is_unique_column(table_name, child_column) else '∞'
        dot_lines.append(
            f'  {dot_id(parent_table)} -> {dot_id(table_name)} '
            f'[label="{child_column} -> {parent_column}", taillabel="1", headlabel="{child_cardinality}"];'
        )
dot_source = '\n'.join(dot_lines + ['}']) + '\n'
ERD_DOT.write_text(dot_source, encoding='utf-8')
if shutil.which('dot'):
    try:
        import graphviz
        graphviz.Source(dot_source).render(filename=str(ERD_SVG.with_suffix('')), format='svg', cleanup=True)
        graphviz.Source(dot_source).render(filename=str(ERD_PNG.with_suffix('')), format='png', cleanup=True)
        print(f'Created {ERD_SVG.resolve()}')
        print(f'Created {ERD_PNG.resolve()}')
    except Exception as error:
        print(f'Graphviz rendering failed: {error}')
else:
    print('Graphviz dot executable not found; use the DOT ERD source or install Graphviz to render SVG and PNG.')
print(f'Created {ERD_DOT.resolve()}')

Created /workspaces/surf_equipment/output/ERD/surf_equipment_erd.svg
Created /workspaces/surf_equipment/output/ERD/surf_equipment_erd.png
Created /workspaces/surf_equipment/output/ERD/surf_equipment_erd.dot


### Task 5: Build fact sheets

This section uses SQL queries to retrieve every board and rig, then formats a fact sheet for each item.

In [5]:
boards_fact_sheet_sql = """
SELECT
    boards.*,
    COALESCE(
        (
            SELECT GROUP_CONCAT(rig_id, ', ')
            FROM (
                SELECT DISTINCT rig_id
                FROM rig_board
                WHERE rig_board.board_id = boards.board_id
                ORDER BY rig_id
            )
        ),
        ''
    ) AS rig_ids
FROM boards
ORDER BY boards.board_id;
"""

boards_data = pd.read_sql_query(boards_fact_sheet_sql, connection)
print(f'Loaded {len(boards_data)} boards for fact sheets.')

Loaded 4 boards for fact sheets.


#### Task 5a: Build board fact sheets

This cell uses the SQL board query to create one fact sheet for every board, including its compatible rigs, using the same field/value format as the original board fact sheet.

In [6]:
def style_fact_sheet(fact_sheet, caption):
    return (
        fact_sheet.style
        .hide(axis='index')
        .set_caption(caption)
        .set_properties(subset=['Field'], **{
            'font-weight': 'bold',
            'background-color': '#DDEBF7',
            'text-align': 'left',
            'vertical-align': 'top',
            'white-space': 'nowrap',
        })
        .set_properties(subset=['Value'], **{
            'text-align': 'left',
            'vertical-align': 'top',
            'white-space': 'pre-wrap',
        })
        .set_table_styles([
            {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('text-align', 'left')]},
            {'selector': 'th', 'props': [('background-color', '#5B9BD5'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('border-bottom', '1px solid #D9E2F3'), ('text-align', 'left')]},
        ])
    )


board_fact_sheets = {}
for _, board_report in boards_data.iterrows():
    board_id = board_report['board_id']
    board_fact_sheet = (
        board_report.rename('Value')
        .rename_axis('Field')
        .reset_index()
    )
    board_fact_sheet.loc[board_fact_sheet['Field'].eq('rig_ids'), 'Field'] = 'compatible_rigs'
    board_fact_sheets[board_id] = style_fact_sheet(
        board_fact_sheet,
        f'Board fact sheet: {board_id}',
    )

print(f'Created {len(board_fact_sheets)} board fact sheets.')

Created 4 board fact sheets.


#### Task 5b: Build rig fact sheets

This cell uses the SQL rig query to create one fact sheet for every rig, including its compatible boards, using the same field/value format as the original rig fact sheet.

In [ ]:
rig_select_columns = [
    f'rigs.{quote_identifier(column)} AS {quote_identifier(column)}'
    for column in tables['rigs'].columns
]
rig_joins = []
master_select_columns = []
for foreign_key_column in tables['rigs'].columns[1:]:
    parent_table = primary_key_to_table.get(foreign_key_column)
    if not parent_table or parent_table == 'rigs':
        continue
    parent_alias = quote_identifier(parent_table)
    rig_joins.append(
        f'LEFT JOIN {quote_identifier(parent_table)} AS {parent_alias} '
        f'ON rigs.{quote_identifier(foreign_key_column)} = '
        f'{parent_alias}.{quote_identifier(primary_keys[parent_table])}'
    )
    master_select_columns.extend(
        f'{parent_alias}.{quote_identifier(column)} AS {quote_identifier(f"{parent_table}.{column}")}'
        for column in tables[parent_table].columns
    )

rig_fact_sheet_sql = f"""
SELECT
    {', '.join(rig_select_columns + master_select_columns)},
    COALESCE(
        (
            SELECT GROUP_CONCAT(board_id, ', ')
            FROM (
                SELECT DISTINCT board_id
                FROM rig_board
                WHERE rig_board.rig_id = rigs.rig_id
                ORDER BY board_id
            )
        ),
        ''
    ) AS compatible_boards
FROM rigs
{' '.join(rig_joins)}
ORDER BY rigs.rig_id;
"""

rigs_data = pd.read_sql_query(rig_fact_sheet_sql, connection)
rig_fact_sheets = {}
for _, rig_report in rigs_data.iterrows():
    rig_id = rig_report['rig_id']
    rig_fact_sheet = (
        rig_report.rename('Value')
        .rename_axis('Field')
        .reset_index()
    )
    rig_fact_sheets[rig_id] = style_fact_sheet(
        rig_fact_sheet,
        f'Rig fact sheet: {rig_id}',
    )

print(f'Created {len(rig_fact_sheets)} rig fact sheets.')

Field,Value
rig_id,8 meter rig
sail_id,8 meter
mast_id,460-80
base_pulley_id,Chique pulley
base_pulley_lenght_setting,?
extension_id,None
extension_length_setting,None
boom_id,Italian boom
boom_length_setting,None
rdm_sdm_shim_id,None


### Task 6: Export the fact sheets and finish the workflow

This section exports the formatted fact sheets as PNG images and closes the database connection.

#### Task 6a: Export all fact sheets

This cell uses Playwright to render every styled board and rig fact sheet as a PNG image.

In [ ]:
import os
from pathlib import Path

from playwright.async_api import async_playwright

BOARD_FACT_SHEETS_DIR = Path('output') / 'boards'
RIG_FACT_SHEETS_DIR = Path('output') / 'rigs'
BOARD_FACT_SHEETS_DIR.mkdir(parents=True, exist_ok=True)
RIG_FACT_SHEETS_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_BROWSER_LIBS = Path('.browser-libs/usr/lib/x86_64-linux-gnu')
os.environ['LD_LIBRARY_PATH'] = f"{LOCAL_BROWSER_LIBS.resolve()}:{os.environ.get('LD_LIBRARY_PATH', '')}"


def output_filename(identifier):
    return ''.join(character if character.isalnum() else '_' for character in str(identifier)).strip('_').lower()


async def export_styler_png(styled_report, output_path):
    html = f"""
    <!doctype html>
    <html>
      <head>
        <meta charset="utf-8">
        <style>
          body {{ margin: 0; background: white; }}
        </style>
      </head>
      <body>{styled_report.to_html()}</body>
    </html>
    """
    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch()
        page = await browser.new_page(device_scale_factor=2)
        await page.set_content(html)
        await page.locator('table').screenshot(path=str(output_path), type='png')
        await browser.close()


for board_id, styled_board_fact_sheet in board_fact_sheets.items():
    output_path = BOARD_FACT_SHEETS_DIR / f'{output_filename(board_id)}.png'
    await export_styler_png(styled_board_fact_sheet, output_path)
    print(f'Created {output_path.resolve()}')

for rig_id, styled_rig_fact_sheet in rig_fact_sheets.items():
    output_path = RIG_FACT_SHEETS_DIR / f'{output_filename(rig_id)}.png'
    await export_styler_png(styled_rig_fact_sheet, output_path)
    print(f'Created {output_path.resolve()}')

Created /workspaces/surf_equipment/board_the_ship_fact_sheet.png
Created /workspaces/surf_equipment/rig_8_meter_fact_sheet.png


#### Task 6b: Close the database connection

This final cleanup cell closes the SQLite connection after all reports and exported files have been created.

In [5]:
connection.close()
print('Database connection closed.')

Database connection closed.
